# 6) Dataset class, Subsets, Dataloader class: introduction to batching

This notebook presents the most common approach to prepare an imported dataset for the training and evaluation of a MLP:

- *Dataset class*, defined by methods __init__(self), __len__(self), __getitem__(self, idx)
- *Subsets function*, to extract the training and validation sets from the original one by indexes
- *DataLoader class* (batching), to create training and validation sets of batching
--------

**The classic approach**: training and evaluation are performed on the whole sets.

- classic split of the original dataset in training and validation sets with sklearn.
- model's training on the whole training set.
- model's validation on the whole validation set.

However, this classic approach is usually computationally expensive, time expensive and memory expensive. 

In practice, the preferred approach consists in creating a 'Dataset class' to access the original dataset; in splitting this Dataset with Subsets into training and validation sets; and in finally performing batching on the training and validation sets through 'Dataloader'. The model is trained on batch, instaed than on the whole training and validation sets, which brings several advantages.

>**The practical favorable approach**: 
>
>- 1) create a '**Dataset class**' which gives access/imports an existing dataset. This class is defined by 3 methods:
>  
>     - __init__(self): returns X, y from the dataset;
>     - __len__(self): returns length of dataset;
>     - __getitem__(self): returns the data at the idx position.
>
>- 2) split Dataset into training and validation sets.
>
>     - split indexes of 'len(self)' into training + validation;
>     - extract training and validation sets from the 'Dataset', by indexes, using '**Subsets**'.
>
>- 3) - create a '**Dataloader class**': returns an iterable object over batches. The batches are selected randomly uniformly and of the same size. Then, define dataloader_training_set and dataloader_validation_set.
>
>- 4) train the MLP model on the dataloader_training_set with batches.
>
>- 5) evaluate the MLP model on the dataloader_validation_set with batches.


In [ ]:
import torch

# import dataset, dataloader, subset
from torch.utils.data import Dataset, DataLoader, Subset

## 1) Dataset

>Creation of a whole *Dataset class* from MNIST, defined from the methods:
>
>- __init__(self) : return data from MNIST
>- __len__(self) : return length of dataset
>- __getitem__(self, idx : int) : return element of index idx

In [2]:
from torchvision.datasets import MNIST

In [4]:
class MNISTDataset(Dataset):
    
    def __init__(self):                   # data from MNIST
        
        data = MNIST('mnist', download = True)
        
        self.images, self.labels = data.data, data.targets
        
    def __len__(self):                   # length of dataset
        
        return len(self.images)
    
    def __getitem__(self, idx : int):    # element of index idx
        
        image = self.images[idx].float().view(28*28)/255.0
        
        label = self.labels[idx]
        
        return image, label
        

In [5]:
# CALL THE DATASET CLASS

dataset = MNISTDataset()


## 2) Subsets

> *Subsets* is a command extracting from the Dataset a subset of data (training or validation sets usually), selected by giving the indexes of elements.

> **Why to use Subsets to extract the training and validation sets from the Dataset?** *It saves the memory because avoids to copy data*. This approach is usually recommended instaed than the Sklearn split, because it is time and memory saving.

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
# PARTITION the INDEXES for TRAINING AND VALIDATION
idx_train, idx_valid = train_test_split(range(len(dataset)), test_size=0.1, random_state=0)

# extract TRAINING SET
train_dataset = Subset(dataset, idx_train)  # train_dataset[i] = dataset[idx_train[i]]

# extract VALIDATION SET
valid_dataset = Subset(dataset, idx_valid)  # valid_dataset[i] = dataset[idx_valid[i]]


## 3) DataLoader

> *DataLoader* is a command splitting the training and/or validation datasets into batch, uniformly selected and of the same size. 
> - It returns an iterable object over the batches, i.e. an object you can only go through with batches.
> - Then, the training and the evaluation phases of the model are performed on batches, instaed than on the whole dataset.
>

> **Why to use Dataloader (i.e. batching) instaed than a classic dataset for training and evaluation?**: *Training and evaluation are performed on small batches, instaed than the whole dataset: a thing that lowers the computational cost.* With real data, the DataLoader approach is preferred and employing the whole dataset is usually computationally infeasible.


**Training and evaluation with DataLoader:**

*Fewer epochs are needed*

-> *The training phase*:

- Loop over batches from the dataloader_training.
- Train the model on each batch.

-> *The evaluation phase*:

- Loop over batches from the dataloader_validation.
- Prediction on each batch.
- Extract target value on each batch. 

> **ADVANTAGES OF BATCHING**:
> 1) Less memory required.
> 2) Gradually processing of data.
> 3) Processes can be parallelized and made faster, i.e. GPUs can be used.
>
>     Rmk) If $n$ GPUs are avaiable, set num_workers at least $n/2$, to achieve fast results.
>
> 4) Faster training and evaluation.
> 5) The selection of batches introduces randomness, which helps to skip local minima and flat regions during the optimization phase.

> **THE BATCH SIZE**: batch are randomly extracted and all of the same size.
> 1) *Small batch*: noisy training, because estimations on too small sets can vary considerably one from another.
> 2) *Large batch*: memory experisive, because GPUs have a limited amount of memory. Furthermore, the complexity might equal that of a classic training, making batching useless.

> **What is an adequate batch size?** A tradeoff between too small and too large. It's chosen as an hyperparameter before training the model.


In [8]:
# OPTIONAL ARGUMENT in DataLoader object,
#
# TELLING HOW TO FORM THE BATCH
# 
# The default is to stock data as Torch tensors, which is exactly what this function does

def collate(batch):

    images = []                             # list of tensors
    labels = []                             # list of tensors

    for image, label in batch:
        images.append(image)
        labels.append(label)

    images = torch.stack(images, dim=0)     # make it a tensor
    labels = torch.tensor(labels)           # make it a tensor

    return images, labels                   # return images and labels as tensors


In [9]:
# DEFINE DATALOADER FOR TRAINING, BATCHING : returns an iteratable object of batches

train_dataloader = DataLoader(

    train_dataset,            # MUST BE SET

    batch_size = 32,          # MUST BE SET
    
    shuffle = True,           # change order to batch at each epoch

    num_workers = 0,          # number of parallel processes to set if training on GPU
                              # = 0 on CPU

    drop_last = True,         # if the batch size does not divide exactly the dataset size,
                              # this command deletes the remaining part

    collate_fn = collate      # OPTIONAL, tells how to form the batch
                              # default: stack data as Torch Tensors
)

In [11]:
# EXAMPLE: READING FIRST ELEMENT OF DATALOADER
# 
# extract the first batch
# from train_dataloader which is an iterable object !!

for batch in train_dataloader:
    break

# OR as an alternative:

#batch = next( iter(train_dataloader) ) 

print("Number of batches:", len(train_dataloader))      # number of batches
print("Images shape:", batch[0].shape)                  # images
print("Labels shape:", batch[1].shape)                  # labels


Number of batches: 1687
Images shape: torch.Size([32, 784])
Labels shape: torch.Size([32])


In [ ]:
# DEFINE DATALOADER FOR VALIDATION, BATCHING : iteratable object of batches

# as done for the training set

valid_dataloader = DataLoader(
    valid_dataset,

    batch_size = 32,

    shuffle = False,

    num_workers = 0,

    drop_last = True,
    
    collate_fn = collate
)

## 4) 5) Example: training and evaluating a MLP classifier with batching

- Create the MLP as usual.
- Training on dataloader_training_set.
- Validation on dataloader_validation_set.

In [13]:
# DEFINE MODEL MLP

class TorchMLP(torch.nn.Module):
    
    def __init__(self, input_dim, hidden_dim, num_classes):
        
        super().__init__()   

        all_dims = [input_dim] + list(hidden_dim) + [num_classes]

        layers = []
        
        for dim1, dim2 in zip(all_dims[:-1], all_dims[1:]):     # define linear layers + ReLu on hidden layers
            
            layers.append(torch.nn.Linear(dim1, dim2))
            
            layers.append(torch.nn.ReLU())

        layers = layers[:-1]

        self.model = torch.nn.Sequential(*layers)    # sequential computations

        
    def forward(self, x):
        
        return self.model(x)
    

In [14]:
# CALL MODEL MLP

model_MLP = TorchMLP(784, [100, 200, 100], 10)

loss_MLP = torch.nn.CrossEntropyLoss()

optimizer_MLP = torch.optim.Adam(model_MLP.parameters(), lr = 0.002)

In [15]:
import torchinfo

torchinfo.summary(model_MLP)

Layer (type:depth-idx)                   Param #
TorchMLP                                 --
├─Sequential: 1-1                        --
│    └─Linear: 2-1                       78,500
│    └─ReLU: 2-2                         --
│    └─Linear: 2-3                       20,200
│    └─ReLU: 2-4                         --
│    └─Linear: 2-5                       20,100
│    └─ReLU: 2-6                         --
│    └─Linear: 2-7                       1,010
Total params: 119,810
Trainable params: 119,810
Non-trainable params: 0

In [17]:
from sklearn.metrics import accuracy_score

In [18]:
# TRAINING AND EVALUATION WITH BATCHING

for epoch in range(10):
  
  # TRAINING ON EACH BATCH
  for images, label in train_dataloader:      # <-- training on each batch!!
    
    optimizer_MLP.zero_grad()                                     
    y_pred = model_MLP(images)                # prediction on batch   
    loss_result = loss_MLP(y_pred, label)     # loss on batch
    loss_result.backward()                                
    optimizer_MLP.step()       


  # EVALUATION ON EACH BATCH
  all_preds = []                # collect prediction on batches
  all_labels = []               # collect labels on batches
  
  with torch.no_grad():                                   # delete gradients, to save memory

      for images, labels in valid_dataloader:             # <-- evaluation on each batch!!     

         logits = model_MLP(images)                       # prediction on each batch

         all_preds.extend(torch.argmax(logits, dim=1)) 

         all_labels.extend(labels)      
                                                          
  print(accuracy_score(all_preds, all_labels))

0.9685828877005348
0.9727606951871658
0.9704211229946524
0.9717580213903744
0.9752673796791443
0.9762700534759359
0.9767713903743316
0.9789438502673797
0.9782754010695187
0.9781082887700535
